# Heteroscedastic Deep Ensemble for 1D Advection-Diffusion-Reaction

This notebook extends the ADR problem by injecting spatially varying
observation noise. The model predicts both the mean response and the
aleatoric noise level, while ensemble spread captures epistemic uncertainty.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'sciml':
    PROJECT = ROOT.parents[1]
else:
    PROJECT = ROOT
SRC = PROJECT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

from deepuq.methods import HeteroscedasticDeepEnsembleRegressor
from deepuq.models import MLP

torch.manual_seed(11)
random.seed(11)

## Noise model

The clean ADR solution follows the same PDE as the previous notebook, but we
observe

$$
y(x) = u(x) + \varepsilon(x), \qquad \varepsilon(x)\sim\mathcal N(0, \sigma^2(x)).
$$

The observation noise is intentionally larger near the center of the domain.

In [ ]:
def solve_adr_1d(params, n_grid=128):
    a, b, c, diff, adv, react = params
    x = torch.linspace(0.0, 1.0, n_grid)
    h = float(x[1] - x[0])
    interior = n_grid - 2
    main = torch.full((interior,), 2.0 * diff / h**2 + react)
    lower = torch.full((interior - 1,), -diff / h**2 - adv / (2.0 * h))
    upper = torch.full((interior - 1,), -diff / h**2 + adv / (2.0 * h))
    A = torch.diag(main) + torch.diag(lower, diagonal=-1) + torch.diag(upper, diagonal=1)
    source = a * torch.sin(math.pi * b * x[1:-1]) + c * torch.cos(2.0 * math.pi * x[1:-1])
    interior_u = torch.linalg.solve(A, source)
    u = torch.zeros_like(x)
    u[1:-1] = interior_u
    return x, u


def sample_params(n, *, ood=False):
    rows = []
    for _ in range(n):
        if ood:
            rows.append((random.uniform(1.2, 1.8), random.uniform(3.5, 5.0), random.uniform(-0.5, 0.5), random.uniform(0.04, 0.09), random.uniform(0.8, 1.2), random.uniform(0.4, 0.8)))
        else:
            rows.append((random.uniform(0.7, 1.3), random.uniform(1.0, 3.0), random.uniform(-0.25, 0.25), random.uniform(0.06, 0.12), random.uniform(0.2, 0.8), random.uniform(0.1, 0.5)))
    return rows


def noise_sigma(x, params):
    _, _, c, _, _, _ = params
    center_bump = 0.01 + 0.05 * torch.exp(-((x - 0.5) ** 2) / 0.02)
    return center_bump + 0.01 * abs(c)


def make_dataset(param_rows, n_grid=128):
    feats = []
    noisy_targets = []
    curves = []
    for params in param_rows:
        x, clean = solve_adr_1d(params, n_grid)
        sigma = noise_sigma(x, params)
        noisy = clean + sigma * torch.randn_like(clean)
        param_tensor = torch.tensor(params, dtype=torch.float32).repeat(n_grid, 1)
        feats.append(torch.cat([x.unsqueeze(1), param_tensor], dim=1))
        noisy_targets.append(noisy.unsqueeze(1))
        curves.append((x, clean, noisy, sigma, params))
    return torch.cat(feats, dim=0), torch.cat(noisy_targets, dim=0), curves

In [ ]:
train_x, train_y, _ = make_dataset(sample_params(80))
test_x, test_y, test_curves = make_dataset(sample_params(20))
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=256, shuffle=True)
ensemble = HeteroscedasticDeepEnsembleRegressor([MLP(7, [64, 64, 64], 2, p_drop=0.0) for _ in range(5)])
ensemble.fit(loader, epochs=80, lr=1e-3, weight_decay=1e-5, seed=33)

In [ ]:
x, clean, noisy, sigma_true, params = test_curves[0]
feats = torch.cat([x.unsqueeze(1), torch.tensor(params, dtype=torch.float32).repeat(x.numel(), 1)], dim=1)
uq = ensemble.predict_uq(feats)
mean = uq.mean.squeeze(-1)
alea = uq.aleatoric_var.sqrt().squeeze(-1)
total = uq.total_var.sqrt().squeeze(-1)

plt.figure(figsize=(8, 4))
plt.scatter(x, noisy, s=8, alpha=0.3, label='noisy obs')
plt.plot(x, clean, label='clean truth')
plt.plot(x, mean, label='ensemble mean')
plt.fill_between(x.numpy(), (mean - 2 * total).numpy(), (mean + 2 * total).numpy(), alpha=0.2, label='total ±2 std')
plt.fill_between(x.numpy(), (mean - 2 * alea).numpy(), (mean + 2 * alea).numpy(), alpha=0.15, label='aleatoric ±2 std')
plt.legend(); plt.tight_layout()